# 🐄 Livestock Tracking — Nostradamus IoTO SDK

Cattle health and GPS tracking: body temperature, heart rate, activity level, rumination, step count, ambient temperature, position and collar battery.

This notebook mirrors `examples/agriculture/livestock_tracking.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **5 animals × 24 hourly readings** (120 records) to the collection
`livestock_sensors`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `livestock_sensors` collection with a schema
2. Generate and send 5 animals × 24 hourly readings
3. Query fever, illness, heat stress and body-temperature ranking
4. Aggregate body temperature, heart rate and rumination statistics
5. Delete a 12-hour window for `CATTLE_001`
6. Delete the collection

## What this notebook highlights

- Fever detection (body temperature > 39.5 °C)
- Illness detection (low activity + low rumination)
- Heat stress alerts (ambient > 30 °C with elevated heart rate)

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🐄 Nostradamus IoTO SDK — Livestock Tracking")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the animal collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/agriculture/livestock_tracking.py`.

In [ ]:
COLLECTION_NAME = "livestock_sensors"

COLLECTION_SCHEMA = {
                    "key": "CATTLE_001",
                    "timestamp": "2025-06-17T10:30:00Z",
                    "latitude": 40.005,
                    "longitude": 22.005,
                    "body_temperature_c": 38.5,
                    "heart_rate_bpm": 60,
                    "activity_level": 5,
                    "rumination_min": 35,
                    "steps_count": 200,
                    "ambient_temperature_c": 22.0,
                    "battery_pct": 85.0,
                }

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Cattle health and GPS tracking data",
            tags=["livestock", "cattle", "health", "gps"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send cattle health data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_livestock_data()` is the
same generator used by the script.

In [ ]:
def generate_livestock_data(num_animals=5, num_readings=24):
    """Generate livestock health and location data.

    Args:
        num_animals: Number of cattle to simulate
        num_readings: Hourly readings per animal over 24h

    Returns:
        List of sensor readings
    """
    data = []
    base_time = datetime.now() - timedelta(hours=24)

    for animal_num in range(1, num_animals + 1):
        animal_id = f"CATTLE_{animal_num:03d}"
        base_lat = 40.003 + animal_num * 0.001
        base_lon = 22.004 + animal_num * 0.001

        for hour in range(num_readings):
            timestamp = base_time + timedelta(hours=hour)
            h = timestamp.hour

            is_resting = h < 6 or h > 21
            ambient_temp = 15 + 10 * max(0, 1.0 - abs(h - 14) / 8.0)
            ambient_temp += random.uniform(-3, 3)
            ambient_temp = round(min(35, max(5, ambient_temp)), 1)

            body_temp = 38.5 + random.uniform(-0.5, 1.0)
            if ambient_temp > 30:
                body_temp += 0.5
            body_temp = round(min(41.0, max(37.5, body_temp)), 1)

            activity = random.randint(0, 3) if is_resting else random.randint(3, 10)
            heart_rate = 50 + activity * 5 + random.randint(-5, 10)
            heart_rate = min(100, max(40, heart_rate))

            rumination = (
                random.randint(20, 50) if not is_resting else random.randint(30, 60)
            )
            if activity > 7:
                rumination = random.randint(0, 15)
            rumination = min(60, max(0, rumination))

            steps = activity * 40 + random.randint(-20, 50)
            steps = max(0, min(500, steps))

            lat = base_lat + random.uniform(-0.002, 0.002)
            lon = base_lon + random.uniform(-0.002, 0.002)

            reading = {
                "key": animal_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "latitude": round(lat, 6),
                "longitude": round(lon, 6),
                "body_temperature_c": body_temp,
                "heart_rate_bpm": heart_rate,
                "activity_level": activity,
                "rumination_min": rumination,
                "steps_count": steps,
                "ambient_temperature_c": ambient_temp,
                "battery_pct": round(
                    100 - (hour / num_readings) * 40 + random.uniform(-5, 5),
                    1,
                ),
            }
            data.append(reading)

    return data

data = generate_livestock_data()
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **🔍 CATTLE_001 readings**
- **🌡️ Fever (body temp > 39.5 °C)**
- **😴 Illness risk (activity < 2 and rumination < 10 min)**
- **🥵 Heat stress (ambient > 30 °C and heart rate > 80 bpm)**
- **Top 5 body temperatures**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # 🔍 CATTLE_001 readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "CATTLE_001"},
        ],
    )
    summary(result, "🔍 CATTLE_001 readings")

    # 🌡️ Fever (body temp > 39.5 °C)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "body_temperature_c", "operator": "gt", "property_value": 39.5},
        ],
    )
    summary(result, "🌡️ Fever (body temp > 39.5 °C)")

    # 😴 Illness risk (activity < 2 and rumination < 10 min)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "activity_level", "operator": "lt", "property_value": 2},
            {"property_name": "rumination_min", "operator": "lt", "property_value": 10},
        ],
    )
    summary(result, "😴 Illness risk (activity < 2 and rumination < 10 min)")

    # 🥵 Heat stress (ambient > 30 °C and heart rate > 80 bpm)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "ambient_temperature_c", "operator": "gt", "property_value": 30},
            {"property_name": "heart_rate_bpm", "operator": "gt", "property_value": 80},
        ],
    )
    summary(result, "🥵 Heat stress (ambient > 30 °C and heart rate > 80 bpm)")

    # Top 5 body temperatures
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "body_temperature_c"],
        order_by='{"field": "body_temperature_c", "order": "desc"}',
        limit=5,
    )
    print("🏆 Top 5 body temperatures")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average body temperature (°C)", "body_temperature_c", StatOperation.AVG),
        ("Maximum heart rate (bpm)", "heart_rate_bpm", StatOperation.MAX),
        ("Minimum rumination (min)", "rumination_min", StatOperation.MIN),
        ("Distinct animals", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one animal

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`CATTLE_001` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "CATTLE_001"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")